In [1]:
from db import get_con
import numpy as np
import pandas as pd

import plotly.express as px

from scipy.stats import ttest_ind, norm, chisquare
from statsmodels.stats.multitest import multipletests
con = get_con()

Setting up database tables...
Done. All tables are ready.


# Описание задачи

Мы провели A/B-тест.
<br>
Результаты лежат в файле Файл <i>task_3_data.csv</i>
<br>
<br>
Группа a - дефолтный вариант.
Группа b - с добавлением нового партнера.
<br>
Ключевая метрика - profit.
<br><br>
Вопросы:
- Проанализируйте и интерпретируйте результаты.
- Какие рекомендации по проведению эксперимента могли бы дать?

# Изучение данных

In [2]:
con.sql(
    """
    select cnt_groups,
           count(distinct uid) as cnt_users
    from (select uid,
                 count(distinct exp_group) as cnt_groups
          from task_3_data
          group by 1) as rd
    group by cnt_groups
    """).df()

,cnt_groups,cnt_users
0,2,204
1,1,199796


In [3]:
con.sql(
    """
    with cte_groups_2 as (
        select uid,
               count(distinct exp_group) as cnt_groups
        from task_3_data
        group by 1)

    select exp_group,
           if(profit > 0, 'Более 0', 'Значение строго = 0') as profit_group,
           count(distinct uid) as cnt_users
    from task_3_data
    where uid not in (select uid
                      from cte_groups_2
                      where cnt_groups > 1)
    group by 1, 2
    order by 1, 2
    """).df()

,exp_group,profit_group,cnt_users
0,a,Более 0,7378
1,a,Значение строго = 0,92862
2,b,Более 0,8088
3,b,Значение строго = 0,91468


In [4]:
con.sql(
    """
    with cte_groups_2 as (
        select uid,
               count(distinct exp_group) as cnt_groups
        from task_3_data
        group by 1)

    select *
    from task_3_data
    where uid not in (select uid
                      from cte_groups_2
                      where cnt_groups > 1)
      and profit < 0
    """).df()

,uid,exp_group,profit


In [5]:
srm_counts = con.sql(
    """
    with cte_groups_2 as (
        select uid,
               count(distinct exp_group) as cnt_groups
        from task_3_data
        group by 1)

    select exp_group,
           count(distinct uid) as cnt_users
    from task_3_data
    where uid not in (select uid
                      from cte_groups_2
                      where cnt_groups > 1)
    group by 1
    """).df()

In [6]:
srm_counts

,exp_group,cnt_users
0,b,99556
1,a,100240


In [7]:
# Гарантируем порядок групп a, b
srm_counts = (
    srm_counts
    .set_index("exp_group")
    .reindex(["a", "b"], fill_value=0)
    .reset_index()
)

total_users = srm_counts["cnt_users"].sum()

In [8]:
srm_counts

,exp_group,cnt_users
0,a,100240
1,b,99556


In [9]:
expected_users = total_users / 2
srm_counts["expected_users"] = expected_users
srm_counts["observed_share_pct"] = (
    100 * srm_counts["cnt_users"] / total_users
)

srm_counts["expected_share_pct"] = 50.0

In [10]:
srm_counts

,exp_group,cnt_users,expected_users,observed_share_pct,expected_share_pct
0,a,100240,99898.0,50.171175,50.0
1,b,99556,99898.0,49.828825,50.0


In [11]:
chi2_stat, p_value = chisquare(
    f_obs=srm_counts["cnt_users"],
    f_exp=srm_counts["expected_users"]
)

alpha = 0.05

print(f"Всего пользователей: {total_users:,}")
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < alpha:
    print(
        "Обнаружен SRM: фактическое распределение статистически значимо отличается от 50/50."
    )
else:
    print(
        "SRM не обнаружен: нет оснований считать, что распределение отличается от 50/50."
    )

Всего пользователей: 199,796
Chi-square statistic: 2.3417
p-value: 0.1260
SRM не обнаружен: нет оснований считать, что распределение отличается от 50/50.


Какие выводы по данным (именно проверка качества данных)?
1. У нас есть 204 пользователя, которые попали в обе группы. Для дальнейшего анализа мы их исключили
2. Остальные пользователи распределились между группами достаточно равномерно, SRM не обнаружено
3. В данных profit нет пропусков, нет profit < 0, но при этом достаточно большое количество пользователей не обладает профитом.

# Основной код

## Изучаем данные по группам

In [12]:
con.execute(
    """
    create or replace temporary view task_3_clean as

    select
        uid,
        min(exp_group) as exp_group,
        sum(profit) as profit
    from task_3_data
    group by uid
    having count(distinct exp_group) = 1
    """
)

In [13]:
con.sql(
    """
    select
        count(*) as rows_count,
        count(distinct uid) as unique_users
    from task_3_clean
    """
).df()

,rows_count,unique_users
0,199796,199796


### Конверсия в пользователя с profit > 0

In [14]:
conversion_by_group = con.sql(
    """
    select
        exp_group,
        count(*) as total_users,

        count(*) filter (
            where profit > 0
        ) as users_with_profit,

        100.0 * count(*) filter (
            where profit > 0
        ) / count(*) as conversion_pct

    from task_3_clean
    group by exp_group
    order by exp_group
    """
).df()

conversion_by_group.round(2)

,exp_group,total_users,users_with_profit,conversion_pct
0,a,100240,7378,7.36
1,b,99556,8088,8.12


In [15]:
fig = px.bar(
    conversion_by_group,
    x="exp_group",
    y="conversion_pct",
    color="exp_group",
    text_auto=".2f",
    title="Конверсия в пользователя с profit > 0",
    labels={
        "exp_group": "Экспериментальная группа",
        "conversion_pct": "Конверсия, %"
    },
    category_orders={
        "exp_group": ["a", "b"]
    }
)

fig.show()

### Средний profit

In [16]:
profit_by_group = con.sql(
    """
    select
        exp_group,
        count(*) as total_users,

        avg(profit) as avg_profit_per_user,

        avg(profit) filter (
            where profit > 0
        ) as avg_profit_per_user_with_profit,

        median(profit) filter (
            where profit > 0
        ) as median_profit_per_user_with_profit,

        sum(profit) as total_profit

    from task_3_clean
    group by exp_group
    order by exp_group
    """
).df()

profit_by_group.round(2)

,exp_group,total_users,avg_profit_per_user,avg_profit_per_user_with_profit,median_profit_per_user_with_profit,total_profit
0,a,100240,17.54,238.35,123.16,1758525.10
1,b,99556,16.27,200.23,75.45,1619478.31


In [17]:
fig = px.bar(
    profit_by_group,
    x="exp_group",
    y="avg_profit_per_user",
    color="exp_group",
    text_auto=".2f",
    title="Средний profit на пользователя",
    labels={
        "exp_group": "Экспериментальная группа",
        "avg_profit_per_user": "Средний profit"
    },
    category_orders={
        "exp_group": ["a", "b"]
    }
)

fig.show()

In [18]:
fig = px.bar(
    profit_by_group,
    x="exp_group",
    y="avg_profit_per_user_with_profit",
    color="exp_group",
    text_auto=".2f",
    title="Средний profit среди пользователей с profit > 0",
    labels={
        "exp_group": "Экспериментальная группа",
        "avg_profit_per_user_with_profit": "Средний profit"
    },
    category_orders={
        "exp_group": ["a", "b"]
    }
)

fig.show()

### Распределение profit

Поскольку более 90% пользователей имеют нулевой profit, распределение лучше анализировать отдельно среди пользователей с profit > 0.

In [19]:
profit_distribution = con.sql(
    """
    select
        exp_group,
        count(*) as users_with_profit,
        avg(profit) as mean_profit,
        median(profit) as median_profit,
        stddev_samp(profit) as std_profit,

        quantile_cont(profit, 0.25) as p25,
        quantile_cont(profit, 0.75) as p75,
        quantile_cont(profit, 0.90) as p90,
        quantile_cont(profit, 0.95) as p95,
        quantile_cont(profit, 0.99) as p99,
        quantile_cont(profit, 0.999) as p999,

        max(profit) as max_profit

    from task_3_clean
    where profit > 0
    group by exp_group
    order by exp_group
    """
).df()

profit_distribution.round(2)

,exp_group,users_with_profit,mean_profit,median_profit,std_profit,p25,p75,p90,p95,p99,p999,max_profit
0,a,7378,238.35,123.16,526.11,24.81,264.05,516.35,807.82,2130.22,7755.05,14849.49
1,b,8088,200.23,75.45,584.58,25.91,200.63,451.84,685.24,1833.57,7041.05,27076.22


In [20]:
positive_profit = con.sql(
    """
    select
        uid,
        exp_group,
        profit
    from task_3_clean
    where profit > 0
    """
).df()

positive_profit.head()

,uid,exp_group,profit
0,e301895eb2249149205aa03de06683e5,b,123.626935
1,4d7c0f40e788f8ef68d97cad49277d1b,a,349.673119
2,d98b666fdc1b1f25cb335cfd54960256,b,28.515853
3,78f18f5820ec4ae25316b2c2982fa11e,a,1101.863459
4,091bb5f21c03cd0a9c64e6ba09c76560,b,32.790402


In [21]:
profit_p99 = positive_profit["profit"].quantile(0.99)

profit_for_hist = positive_profit[
    positive_profit["profit"] <= profit_p99
].copy()

print(f"Верхняя граница графика: {profit_p99:.2f}")

Верхняя граница графика: 1948.45


In [22]:
fig = px.histogram(
    profit_for_hist,
    x="profit",
    color="exp_group",
    nbins=60,
    barmode="overlay",
    opacity=0.6,
    histnorm="probability",
    title="Распределение положительного profit до общего P99",
    labels={
        "profit": "Profit",
        "exp_group": "Группа",
        "probability": "Доля наблюдений"
    },
    category_orders={
        "exp_group": ["a", "b"]
    }
)

fig.show()

In [23]:
fig = px.box(
    positive_profit,
    x="exp_group",
    y="profit",
    color="exp_group",
    points="outliers",
    log_y=True,
    title="Распределение положительного profit",
    labels={
        "exp_group": "Экспериментальная группа",
        "profit": "Profit, логарифмическая шкала"
    },
    category_orders={
        "exp_group": ["a", "b"]
    }
)

fig.update_layout(
    showlegend=False
)

fig.show()

Итого
1. Метрикой я бы выбрала средний profit на пользователя. Он отвечает на бизнес-вопрос: сколько в средне приносит один пользователей в каждой из групп.
2. При этом мы можем использовать дополнительные метрики для объснения результата: CR в пользователя с профитом и как изменилось распределение пользователей с профитом.
3. Есть выбросы, но не сказала бы, что их нужно автоматически удалять. Это может быть вполне норм дорогой покупкой.

Как в результате подводим результаты?

|                         Метрика |                 Описание | Критерий |                 Нужна ли поправка? |
|--------------------------------:|-------------------------:|---------:|-----------------------------------:|
|             Avg profit per user |         Основная метрика | Welch t-test + bootstrap CI |                           Не нужна |
|          Конверсия в profit > 0 | Вторичная, для пояснения | Z-test долей | Holm, если делаем формальный вывод |
|       Avg profit при profit > 0 | Вторичная, для пояснения | Welch t-test + bootstrap CI | Holm, если делаем формальный вывод |
| Profit после чистки от выбросов |              Базовый чек | Welch t-test |                           Не нужна |

## Считаем стат.значимость результата

In [24]:
df_clean = con.sql(
    """
    select
        uid,
        exp_group,
        profit
    from task_3_clean
    """
).df()

df_clean.head()

,uid,exp_group,profit
0,26b377dec1659856dc63873d1db6d3b0,a,0.0
1,5f474272e273ff5c417eed494c29eb50,a,0.0
2,d96599b4a2d8dbaeb39f5ec217923c4a,b,0.0
3,9d3323b8689a4c676649aabe5b4a01b9,a,0.0
4,ff2f20fa1652c5a69ec4a90ede788ed1,b,0.0


In [25]:
print(f"Количество строк: {len(df_clean):,}")
print(f"Уникальных пользователей: {df_clean['uid'].nunique():,}")
print(f"Пропущенных значений: {df_clean.isna().sum().sum():,}")

df_clean.groupby("exp_group").size()

Количество строк: 199,796
Уникальных пользователей: 199,796
Пропущенных значений: 0


exp_group
a    100240
b     99556
dtype: int64

In [26]:
summary = (
    df_clean
    .groupby("exp_group")
    .agg(
        total_users=("uid", "count"),
        users_with_profit=(
            "profit",
            lambda values: (values > 0).sum()
        ),
        avg_profit_per_user=("profit", "mean"),
        total_profit=("profit", "sum")
    )
    .reset_index()
)

positive_profit_summary = (
    df_clean[df_clean["profit"] > 0]
    .groupby("exp_group")
    .agg(
        avg_profit_with_profit=("profit", "mean"),
        median_profit_with_profit=("profit", "median")
    )
    .reset_index()
)

summary = summary.merge(
    positive_profit_summary,
    on="exp_group"
)

summary["conversion_pct"] = (
    100
    * summary["users_with_profit"]
    / summary["total_users"]
)

summary.round(2)

,exp_group,total_users,users_with_profit,avg_profit_per_user,total_profit,avg_profit_with_profit,median_profit_with_profit,conversion_pct
0,a,100240,7378,17.54,1758525.10,238.35,123.16,7.36
1,b,99556,8088,16.27,1619478.31,200.23,75.45,8.12


In [27]:
profit_a = df_clean.loc[
    df_clean["exp_group"] == "a",
    "profit"
].to_numpy()

profit_b = df_clean.loc[
    df_clean["exp_group"] == "b",
    "profit"
].to_numpy()

positive_profit_a = profit_a[profit_a > 0]
positive_profit_b = profit_b[profit_b > 0]

### Bootstrap: Функция возвращает разницу B − A и её 95%-й доверительный интервал.

In [28]:
def bootstrap_mean_difference(
    group_a,
    group_b,
    iterations=1000,
    random_state=42
):
    rng = np.random.default_rng(random_state)
    bootstrap_differences = np.empty(iterations)

    for i in range(iterations):
        sample_a = rng.choice(
            group_a,
            size=len(group_a),
            replace=True
        )

        sample_b = rng.choice(
            group_b,
            size=len(group_b),
            replace=True
        )

        bootstrap_differences[i] = (
            sample_b.mean() - sample_a.mean()
        )

    difference = group_b.mean() - group_a.mean()

    ci_low, ci_high = np.quantile(
        bootstrap_differences,
        [0.025, 0.975]
    )

    return difference, ci_low, ci_high

### Avg Profit per User: Welch T-test

In [29]:
primary_t_stat, primary_p_value = ttest_ind(
    profit_b,
    profit_a,
    equal_var=False
)

primary_difference = (
    profit_b.mean() - profit_a.mean()
)

primary_lift_pct = (
    100 * primary_difference / profit_a.mean()
)

print(f"Mean A: {profit_a.mean():.2f}")
print(f"Mean B: {profit_b.mean():.2f}")
print(f"Разница B − A: {primary_difference:.2f}")
print(f"Изменение: {primary_lift_pct:.2f}%")
print(f"Welch t-statistic: {primary_t_stat:.3f}")
print(f"p-value: {primary_p_value:.4f}")

Mean A: 17.54
Mean B: 16.27
Разница B − A: -1.28
Изменение: -7.27%
Welch t-statistic: -1.720
p-value: 0.0855


### Avg Profit per User: bootstrap CI

In [30]:
(
    primary_boot_difference,
    primary_ci_low,
    primary_ci_high
) = bootstrap_mean_difference(
    profit_a,
    profit_b,
    iterations=1000,
    random_state=42
)

print(f"Разница B − A: {primary_boot_difference:.2f}")

print(
    "Bootstrap 95% CI: "
    f"[{primary_ci_low:.2f}; {primary_ci_high:.2f}]"
)

Разница B − A: -1.28
Bootstrap 95% CI: [-2.64; 0.28]


### Конверсия в profit > 0: z-test

In [31]:
users_a = len(profit_a)
users_b = len(profit_b)

converted_a = (profit_a > 0).sum()
converted_b = (profit_b > 0).sum()

conversion_a = converted_a / users_a
conversion_b = converted_b / users_b

pooled_conversion = (
    (converted_a + converted_b)
    / (users_a + users_b)
)

standard_error = np.sqrt(
    pooled_conversion
    * (1 - pooled_conversion)
    * (1 / users_a + 1 / users_b)
)

conversion_z_stat = (
    (conversion_b - conversion_a)
    / standard_error
)

conversion_p_value = (
    2 * norm.sf(abs(conversion_z_stat))
)

conversion_difference_pp = (
    100 * (conversion_b - conversion_a)
)

print(f"Conversion A: {conversion_a:.2%}")
print(f"Conversion B: {conversion_b:.2%}")

print(
    "Разница B − A: "
    f"{conversion_difference_pp:.2f} п.п."
)

print(f"Z-statistic: {conversion_z_stat:.3f}")
print(f"p-value: {conversion_p_value:.6g}")

Conversion A: 7.36%
Conversion B: 8.12%
Разница B − A: 0.76 п.п.
Z-statistic: 6.387
p-value: 1.69061e-10


### Avg Profit per user with profit > 0: Welch t-test

In [32]:
positive_t_stat, positive_p_value = ttest_ind(
    positive_profit_b,
    positive_profit_a,
    equal_var=False
)

positive_difference = (
    positive_profit_b.mean()
    - positive_profit_a.mean()
)

positive_lift_pct = (
    100
    * positive_difference
    / positive_profit_a.mean()
)

print(f"Mean A: {positive_profit_a.mean():.2f}")
print(f"Mean B: {positive_profit_b.mean():.2f}")
print(f"Разница B − A: {positive_difference:.2f}")
print(f"Изменение: {positive_lift_pct:.2f}%")
print(f"Welch t-statistic: {positive_t_stat:.3f}")
print(f"p-value: {positive_p_value:.6g}")

Mean A: 238.35
Mean B: 200.23
Разница B − A: -38.11
Изменение: -15.99%
Welch t-statistic: -4.268
p-value: 1.98774e-05


### Поправка для вторичных метрик

In [33]:
secondary_tests = pd.DataFrame({
    "metric": [
        "Conversion to profit > 0",
        "Avg profit with profit > 0"
    ],
    "p_value": [
        conversion_p_value,
        positive_p_value
    ]
})

secondary_tests["p_value_holm"] = multipletests(
    secondary_tests["p_value"],
    method="holm"
)[1]

secondary_tests["significant_holm"] = (
    secondary_tests["p_value_holm"] < 0.05
)

secondary_tests

,metric,p_value,p_value_holm,significant_holm
0,Conversion to profit > 0,1.690612e-10,3.381224e-10,True
1,Avg profit with profit > 0,1.987738e-05,1.987738e-05,True


### Проверка после очистки от выбросов
Используем один порог для обеих групп. Значения не удаляем, а ограничиваем сверху

In [34]:
winsorization_limit = df_clean["profit"].quantile(0.99)

df_clean["profit_winsorized"] = (
    df_clean["profit"]
    .clip(upper=winsorization_limit)
)

print(
    "Общий порог P99: "
    f"{winsorization_limit:.2f}"
)


profit_winsorized_a = df_clean.loc[
    df_clean["exp_group"] == "a",
    "profit_winsorized"
].to_numpy()

profit_winsorized_b = df_clean.loc[
    df_clean["exp_group"] == "b",
    "profit_winsorized"
].to_numpy()

winsorized_t_stat, winsorized_p_value = ttest_ind(
    profit_winsorized_b,
    profit_winsorized_a,
    equal_var=False
)

winsorized_difference = (
    profit_winsorized_b.mean()
    - profit_winsorized_a.mean()
)

winsorized_lift_pct = (
    100
    * winsorized_difference
    / profit_winsorized_a.mean()
)

print(f"Mean A: {profit_winsorized_a.mean():.2f}")
print(f"Mean B: {profit_winsorized_b.mean():.2f}")
print(f"Разница B − A: {winsorized_difference:.2f}")
print(f"Изменение: {winsorized_lift_pct:.2f}%")
print(f"p-value: {winsorized_p_value:.6g}")

Общий порог P99: 407.71
Mean A: 11.56
Mean B: 10.77
Разница B − A: -0.79
Изменение: -6.80%
p-value: 0.00134874


### Результаты

По основной метрике
1. A: 17.54
2. B: 16.27
3. Изменение: −1.28, или −7.3%
4. p-value ≈ 0.086
5. Доверительный интервал включает ноль

По основной метрике, мы не можем утверждать, что в эксперименте профит хуже. Однако по вторичным метрикам мы видим следующую картину:
1. В тестовой группе конверсия выросла с 7.36% до 8.12% (результат стат.значим)
2. Профит на пользователя с профитом упал в тестовой группе (результат также значим)

Мы также можем предположить, что разница незначима из-за большой десперсии. После очистки от выбросов — результат сохраняется и становится значим.
Экстремальный значения — максируют снижения в тестовой группе.

Я бы сделала так
1. Не катила бы текущую версию.
2. Поисследовала бы, почему новый партнер генерит менее маржинальные заказы и, судя по метрикам, что идет каннибализация